<font size="+3"><strong> Interactive Dashboard</strong></font>

In [1]:
!kaggle datasets list -s "Airline"

ref                                            title                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------  --------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
iamsouravbanerjee/airline-dataset              Airline Dataset                     13133485  2023-09-26 01:01:28.317000          35117        352  1.0              
teejmahal20/airline-passenger-satisfaction     Airline Passenger Satisfaction       2841945  2020-02-20 16:51:16.547000         137213       1060  0.9411765        
crowdflower/twitter-airline-sentiment          Twitter US Airline Sentiment         2678605  2019-10-16 00:04:05.163000         146328       1174  0.8235294        
eugeniyosetrov/airline-delays                  Airline Delays                        112515  2023-10-10 09:18:17.287000           3900        145  0.9411765        
juhibhojan

### Fetch Dataset

### Unzip and Read the Data

In [2]:
import zipfile

In [3]:
with zipfile.ZipFile("airline-passenger-satisfaction.zip", "r") as file:
    file.extractall("Airline")

##### Read Data

In [4]:
import pandas as pd

df = pd.read_csv("Airline/train.csv")

# Drop the '"Unnamed: 0", "id"' columns
# df.drop(columns=["Unnamed: 0", "id"], inplace=True)
df.drop(columns=["Unnamed: 0", "id"], errors="ignore", inplace=True)

# Encode categorical variables
df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})

df["Customer Type"] = df["Customer Type"].map({"disloyal Customer": 0, "Loyal Customer": 1})

df["Type of Travel"] = df["Type of Travel"].map({"Personal Travel": 0, "Business travel": 1})

df["satisfaction"] = df["satisfaction"].map({"neutral or dissatisfied": 0, "satisfied": 1})

df["Class"] = df["Class"].map({"Eco": 0, "Eco Plus": 1, "Business": 2})

# Median imputation for missing arrival delays
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())

# Save new train data 
df.to_csv("Airline/train_encoded.csv", index=False)

In [5]:
import plotly.express as px
from dash import Input, Output, dcc, html
from dash import Dash
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# Prepare Data

## Import

In [6]:
def wrangle(filepath):  
    """Read APS data file into ``DataFrame``.

    Returns only satisfied passengers whose Flight distance is less than 4,500 miles.

    Parameters
    ----------
    filepath : str
        Location of CSV file.
    """
    # Load Data
    df = pd.read_csv(filepath)
    # Create Mask
    mask = (df["satisfaction"] == 1) & (df["Flight Distance"] < 4500)
    df = df[mask]
    
    return df

In [7]:
df = wrangle("Airline/train_encoded.csv")

print("df type:", type(df))
print("df shape:", df.shape)
df.head()

df type: <class 'pandas.core.frame.DataFrame'>
df shape: (45004, 23)


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
2,1,1,26,1,2,1142,2,2,2,2,...,5,4,3,4,4,4,5,0,0.0,1
4,0,1,61,1,2,214,3,3,3,3,...,3,3,4,4,3,3,3,0,0.0,1
7,1,1,52,1,2,2035,4,3,4,4,...,5,5,5,5,4,5,4,4,0.0,1
13,0,1,33,0,0,946,4,2,4,3,...,4,4,5,2,2,2,4,0,0.0,1
16,1,1,26,1,2,2123,3,3,3,3,...,4,5,3,4,5,4,4,49,51.0,1


In [8]:
import os
os.listdir("Airline")

['test.csv', 'train.csv', 'train_encoded.csv']

# Build Dashboard

## Application Layout

Instantiate a `Dash` application 

In [9]:
app = Dash(__name__)

print("app type:", type(app))

app type: <class 'dash.dash.Dash'>


In [10]:
app.layout = html.Div(
    [
        # Application Title
        html.H1("Airline Passenger Satisfaction"), 
        # Bar chart Element
        html.H2("High Variance Features"),
        # Bar Chart graph
        dcc.Graph(id="bar-chart"),
        dcc.RadioItems(options=[{"label": "trimmed", "value":True}, {"label": "not trimmed", "value":False}], value=True, id="trim-button"),
        #K-Means Slider
        html.H2("K-means Clustering"),
        html.H3("Number of Clusters (k)"),
        dcc.Slider(min=2, max=12, step=1, value=2, id="k-slider"),
        html.Div(id="metrics"),
        # PCA Scatter plot
        dcc.Graph(id="pca-scatter")
    ])

## Variance Bar Chart

Done

### Business Layer

In [11]:
def get_high_var_features(trimmed=True, return_feat_names=True): 
    """Returns the five highest-variance features of ``df``.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    return_feat_names : bool, default=False
        If ``True``, returns feature names as a ``list``. If ``False``
        returns ``Series``, where index is feature names and values are
        variances.
    """
    # Calculate Variance
    if trimmed:
        top_five_features = (df.apply(trimmed_var).sort_values().tail(5))
    else: 
        top_five_features = df.var().sort_values().tail(5) 
        
    # Extract names
    if return_feat_names:
        top_five_features = top_five_features.index.tolist()
        
    return top_five_features

> Done

### Service Layer

In [12]:
@app.callback(Output("bar-chart", "figure"), Input("trim-button", "value"))

def serve_bar_chart(trimmed=True): 
    """Returns a horizontal bar chart of five highest-variance features.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.
    """
    # Get features
    top_five_features = get_high_var_features(trimmed=trimmed, return_feat_names=False)
    
    # Build bar chart
    fig = px.bar(x=top_five_features, y=top_five_features.index, orientation="h")
    fig.update_layout(xaxis_title="Variance", yaxis_title="Feature")
    
    return fig

> Done

BAR CHART: `serve_bar_chart` function to add a bar chart to `"bar-chart"`.
> Done

### Callback Decorator
> Done

In [13]:
def get_model_metrics(trimmed=True, k=2,return_metrics=False):  
    """Build ``KMeans`` model based on five highest-variance features in ``df``.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    k : int, default=2
        Number of clusters.

    return_metrics : bool, default=False
        If ``False`` returns ``KMeans`` model. If ``True`` returns ``dict``
        with inertia and silhouette score.

    """
    # Get high var features
    features= get_high_var_features(trimmed=trimmed, return_feat_names=True)
    # Create feature matrix
    X = df[features]
    # Build model
    model = make_pipeline(StandardScaler(),KMeans(n_clusters=k, random_state=42))
    model. fit(X)

    if return_metrics:
        # Calculate inertia
        i = model.named_steps["kmeans"].inertia_
        # Calculate silhouette score
        ss = silhouette_score(X, model.named_steps["kmeans"].labels_)
        # Put results into dictionary
        metrics = {"inertia": round(i), "silhouette": round(ss, 3)}
        # Return dictionary to user
        return metrics
    
    return model

In [14]:
@app.callback(Output("metrics", "children"), Input("trim-button", "value"), Input("k-slider", "value"))

def serve_metrics(trimmed=True, k=2):  
    """Returns list of ``H3`` elements containing inertia and silhouette score
    for ``KMeans`` model.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    k : int, default=2
        Number of clusters.
    """
    # Get metrics
    metrics = get_model_metrics(trimmed=trimmed, k=k, return_metrics=True)

    # Add metrics to HTML
    text = [html.H3(f"Inertia:{metrics['inertia']}"), html.H3(f"Silhouette Score:{metrics['silhouette']}")]
   
    return text

In [15]:
def get_pca_labels(trimmed=True, k=2):  
    """
    ``KMeans`` labels.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    k : int, default=2
        Number of clusters.
    """
    # Create feature matrix
    features = get_high_var_features(trimmed=trimmed, return_feat_names=True)
    X =df[features]
    # Build transformer
    transformer = PCA(n_components=2, random_state=42)
    # Transform data
    X_t = transformer.fit_transform(X)
    X_pca = pd.DataFrame(X_t, columns=["PC1","PC2"])

    # Add labels
    model =get_model_metrics(trimmed=trimmed, k=k, return_metrics=False)
    X_pca["labels"] = model.named_steps["kmeans"].labels_.astype(str)
    X_pca.sort_values("labels", inplace =True)
    
    return X_pca

In [16]:
@app.callback(Output("pca-scatter", "figure"), Input("trim-button", "value"), Input("k-slider", "value"))

def serve_scatter_plot(trimmed=True, k=2):
    """Build 2D scatter plot of ``df`` with ``KMeans`` labels.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    k : int, default=2
        Number of clusters.
    """
    fig=px.scatter(data_frame=get_pca_labels(trimmed=trimmed, k=k),
              x="PC1", y="PC2", color="labels",title="PCA Representation of Clusters")
    fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
    
    return fig

## K-means Slider and Metrics

In [17]:
app.run(host="127.0.0.1", port=9000, debug=True)

## Author

<a href="https://www.linkedin.com/in/andrew-kalumba-harris/">ANDREW KALUMBA YIGGA</a><br>
<a href =""> </a>


| Date (YYYY-MM-DD) | Prepared By     | 
| ----------------- | --------------  | 
| 2026-07-29        | Author          | 


## <h3 align="center">  Data Science 2026. <h3/>